In [1]:
%pip install ucimlrepo --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from ucimlrepo import fetch_ucirepo
ds = fetch_ucirepo(id=601) # 601 = AI4I 2020 Predictive Maintenance
df_raw =pd.read_csv("../data/ai4i2020.csv")
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
df_raw.head()

Shape: (10000, 14)
Columns: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
# Some versions include units, e.g. "Air temperature [K]". Remove them.
df = df_raw.copy()
df.columns = [c.split(" [")[0].strip() for c in df.columns]
rename_map = {
"Type": "type",
"Air temperature": "air_temp_k",
"Process temperature": "process_temp_k",
"Rotational speed": "rpm",
"Torque": "torque_nm",
"Tool wear": "tool_wear_min",
"Machine failure": "machine_failure",
}
df = df.rename(columns=rename_map)[list(rename_map.values())]
# Drops UDI, Product ID and the failure-mode columns TWF, HDF, PWF, OSF, RNF.
# Those failure modes ARE the failure, so keeping them would leak the answer.
df.head()

,type,air_temp_k,process_temp_k,rpm,torque_nm,tool_wear_min,machine_failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [4]:
print("Missing values per column:")
print(df.isna().sum())
print("\nData types:")
print(df.dtypes)
print("\nTarget balance (share of rows):")
print(df["machine_failure"].value_counts(normalize=True).round(4))

Missing values per column:
type               0
air_temp_k         0
process_temp_k     0
rpm                0
torque_nm          0
tool_wear_min      0
machine_failure    0
dtype: int64

Data types:
type                object
air_temp_k         float64
process_temp_k     float64
rpm                  int64
torque_nm          float64
tool_wear_min        int64
machine_failure      int64
dtype: object

Target balance (share of rows):
0    0.9661
1    0.0339
Name: machine_failure, dtype: float64


In [5]:
df.to_csv("../data/ai4i_clean.csv", index=False)
print("Saved", df.shape, "to data/ai4i_clean.csv")

Saved (10000, 7) to data/ai4i_clean.csv


In [7]:
#connect to your workspace
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
SUBSCRIPTION_ID = "a90ca5d3-6244-43a4-aa4b-bbaf266acc45"
RESOURCE_GROUP = "rg-mlab1-husnainhassan"
WORKSPACE = "mlw-lab1-husnainhassan"
ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID,
RESOURCE_GROUP, WORKSPACE)
print("Connected to:", ml_client.workspace_name)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Connected to: mlw-lab1-husnainhassan


In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
csv_asset = Data(
name="ai4i-maintenance-csv",
version="1",
type=AssetTypes.URI_FILE,
path="../data/ai4i_clean.csv",
description="AI4I 2020 predictive maintenance, cleaned: 6 features +machine_failure",
)
csv_asset = ml_client.data.create_or_update(csv_asset)
print("Registered:", csv_asset.name, "version", csv_asset.version)

In [1]:
cd ~/cloudfiles/code/Users/*/azureml-knn-maintenance
git add data/ai4i_clean.csv notebooks/00_prepare_data.ipynb
git commit -m "Part 4: clean AI4I dataset and data prep notebook"
git push

SyntaxError: invalid decimal literal (1367827645.py, line 2)